# TimesFM — All Forces 2026 Full-Year Forecast (Jan–Dec)

Forecasts Jan–Dec 2026 at **force level** for all 43 forces using TimesFM zero-shot.  
Evaluates against Jan–Mar 2026 actuals (the 3 months currently available).  
Apr–Dec 2026 are shown as forward forecast only.

**Training context:** 2012–2019 + 2022–2025 (pandemic years 2020–2021 excluded)  
**Crime types:** all 16  
**Evaluation window:** Jan–Mar 2026 (actuals available)  
**Forecast window:** Jan–Dec 2026

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import timesfm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display

## 1. Load and aggregate to force level

In [ ]:
CRIME_TYPES = [
    'Violence and sexual offences', 'Criminal damage and arson', 'Drugs',
    'Violent crime', 'Burglary', 'Other theft', 'Vehicle crime',
    'Public order', 'Other crime', 'Shoplifting', 'Robbery',
    'Bicycle theft', 'Theft from the person', 'Possession of weapons',
    'Public disorder and weapons', 'Anti-social behaviour',
]
EXCLUDE_FORCES = {
    'British Transport Police',          # national rail force, funded separately
    'Police Service of Northern Ireland', # different jurisdiction
}
TRAIN_YEARS   = set(range(2012, 2020)) | set(range(2022, 2026))
FORECAST_MONTHS = pd.date_range('2026-01-01', periods=12, freq='MS')  # Jan–Dec 2026
EVAL_MONTHS     = pd.date_range('2026-01-01', periods=3, freq='MS')   # Jan–Mar (actuals available)

raw = pd.read_parquet('../data/processed/crimes_clean_dedup_all_years.parquet')
raw = raw[raw['Crime type'].isin(CRIME_TYPES) & ~raw['Falls within'].isin(EXCLUDE_FORCES)].copy()
raw['month'] = pd.to_datetime(raw['Month'])
raw = raw[raw['month'].dt.year >= 2012]

force_monthly = (
    raw.groupby(['Falls within', 'month'])
    .size()
    .reset_index(name='count')
    .rename(columns={'Falls within': 'force'})
)

all_forces = sorted(force_monthly['force'].unique())
print(f'Forces: {len(all_forces)}')
print(f'Excluded: {EXCLUDE_FORCES}')
print(f'Date range: {force_monthly["month"].min().date()} → {force_monthly["month"].max().date()}')
print(f'Forecast window: {FORECAST_MONTHS[0].date()} → {FORECAST_MONTHS[-1].date()}')
print(f'Evaluation window: {EVAL_MONTHS[0].date()} → {EVAL_MONTHS[-1].date()}')

## 2. Load TimesFM

In [ ]:
print('Loading TimesFM...')
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained('google/timesfm-2.5-200m-pytorch')
tfm.compile(timesfm.ForecastConfig(
    max_context=512,
    max_horizon=128,
    per_core_batch_size=32,
    infer_is_positive=True,
    normalize_inputs=True,
))
print('Ready.')

## 3. Build context series and forecast

In [ ]:
inputs = []
context_info = []

for force in all_forces:
    series = (
        force_monthly[
            (force_monthly['force'] == force) &
            (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
        ]
        .sort_values('month')['count']
        .values
        .astype(float)
    )
    inputs.append(series)
    context_info.append({'force': force, 'context_months': len(series), 'last_value': series[-1] if len(series) > 0 else 0})

context_df = pd.DataFrame(context_info)
print('Context summary:')
display(context_df.set_index('force'))

# Forecast all forces in one batch
print('\nForecasting...')
point_forecast, _ = tfm.forecast(horizon=12, inputs=inputs)
forecasts_arr = np.clip(np.array(point_forecast)[:, :12], 0, None)  # (N, 12)
print('Done.')

# Build tidy forecast dataframe
forecast_rows = []
for i, force in enumerate(all_forces):
    for j, month in enumerate(FORECAST_MONTHS):
        forecast_rows.append({
            'force': force,
            'month': month,
            'forecast': forecasts_arr[i, j],
            'has_actual': month in EVAL_MONTHS,
        })
forecast_df = pd.DataFrame(forecast_rows)

## 4. Attach actuals (Jan–Mar 2026)

In [ ]:
actuals_26 = force_monthly[force_monthly['month'].isin(EVAL_MONTHS)].copy()

forecast_df = forecast_df.merge(
    actuals_26[['force', 'month', 'count']].rename(columns={'count': 'actual'}),
    on=['force', 'month'],
    how='left'
)

# Baseline: 2025 monthly mean per force
baseline = (
    force_monthly[force_monthly['month'].dt.year == 2025]
    .groupby('force')['count'].mean()
    .rename('baseline')
    .reset_index()
)
forecast_df = forecast_df.merge(baseline, on='force', how='left')

print(f'Forecast rows: {len(forecast_df):,}')
print(f'With actuals: {forecast_df["actual"].notna().sum()}')

## 5. Evaluation metrics (Jan–Mar 2026)

In [ ]:
# metric helpers (defined once, reused across the notebook)
def mae(pred, actual):
    return float(np.mean(np.abs(pred - actual)))

def rmse(pred, actual):
    return float(np.sqrt(np.mean((pred - actual) ** 2)))

def mape(pred, actual):
    # guard against division by zero where actual demand is 0
    return float(np.mean(np.abs((pred - actual) / np.where(actual == 0, 1, actual))) * 100)

eval_rows = []
eval_data = forecast_df[forecast_df['actual'].notna()]

for force, g in eval_data.groupby('force'):
    actual_vals   = g['actual'].values.astype(float)
    forecast_vals = g['forecast'].values
    baseline_vals = g['baseline'].values

    eval_rows.append({
        'force':          force,
        'baseline_mae':   mae(baseline_vals, actual_vals),
        'model_mae':      mae(forecast_vals,  actual_vals),
        'baseline_rmse':  rmse(baseline_vals, actual_vals),
        'model_rmse':     rmse(forecast_vals,  actual_vals),
        'model_mape':     mape(forecast_vals,  actual_vals),
        'mean_actual':    actual_vals.mean(),
        'mean_forecast':  forecast_vals.mean(),
    })

eval_df = pd.DataFrame(eval_rows)
eval_df['delta_mae']   = eval_df['baseline_mae'] - eval_df['model_mae']
eval_df['rmae']        = eval_df['model_mae'] / eval_df['baseline_mae']
eval_df['model_wins']  = eval_df['model_mae'] < eval_df['baseline_mae']
eval_df = eval_df.sort_values('model_mape')

print(f'Overall — MAE: {eval_df["model_mae"].mean():.1f}  MAPE: {eval_df["model_mape"].mean():.1f}%  '
      f'Win rate: {eval_df["model_wins"].mean()*100:.0f}%  RMAE: {eval_df["rmae"].mean():.3f}')
print()
display(
    eval_df[['force','mean_actual','mean_forecast','baseline_mae','model_mae',
             'delta_mae','rmae','model_mape','model_wins']]
    .round(1)
    .rename(columns={
        'force':'Force','mean_actual':'Avg Actual','mean_forecast':'Avg Forecast',
        'baseline_mae':'Baseline MAE','model_mae':'TimesFM MAE',
        'delta_mae':'Δ MAE','rmae':'RMAE','model_mape':'MAPE %','model_wins':'Wins'
    })
    .set_index('Force')
)

## 6. MAPE ranking — best to worst performing forces

In [ ]:
fig, ax = plt.subplots(figsize=(10, 12))
colors = ['#4CAF50' if w else '#F44336' for w in eval_df['model_wins']]
bars = ax.barh(eval_df['force'], eval_df['model_mape'], color=colors, edgecolor='white', linewidth=0.5)
ax.axvline(10, color='orange', linestyle='--', linewidth=1, label='10% threshold')
ax.axvline(20, color='red',    linestyle='--', linewidth=1, label='20% threshold')
ax.set_xlabel('MAPE % (lower = better)')
ax.set_title('TimesFM Jan–Mar 2026 Evaluation\nMAPE % by Force (green = beats baseline)', fontweight='bold')
ax.legend(loc='lower right')

# Annotate values
for bar, v in zip(bars, eval_df['model_mape']):
    ax.text(v + 0.3, bar.get_y() + bar.get_height()/2,
            f'{v:.1f}%', va='center', fontsize=7)

plt.tight_layout()
os.makedirs('../outputs', exist_ok=True)
plt.savefig('../outputs/all_forces_mape_ranking.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Forecast vs Actual grid (all forces)

In [ ]:
HISTORY_START = '2023-01-01'
n_forces = len(all_forces)
ncols = 5
nrows = int(np.ceil(n_forces / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(22, nrows * 3.5))
axes_flat = axes.flatten()
fig.suptitle('TimesFM 2026 Forecast (Jan–Jun) vs Actuals (Jan–Mar) — All Forces',
             fontsize=13, fontweight='bold')

for i, force in enumerate(all_forces):
    ax = axes_flat[i]

    # History
    history = force_monthly[
        (force_monthly['force'] == force) &
        (force_monthly['month'] >= HISTORY_START) &
        (force_monthly['month'].dt.year.isin(TRAIN_YEARS))
    ].sort_values('month')
    ax.plot(history['month'], history['count'], color='#94a3b8', linewidth=1, label='History')

    # Full 6-month forecast
    fc = forecast_df[forecast_df['force'] == force].sort_values('month')
    ax.plot(fc['month'], fc['forecast'], color='#2196F3', linewidth=1.8,
            marker='o', markersize=4, linestyle='--', label='Forecast', zorder=5)

    # Actuals (Jan–Mar only)
    fc_eval = fc[fc['actual'].notna()]
    if not fc_eval.empty:
        ax.plot(fc_eval['month'], fc_eval['actual'], color='#FF5722', linewidth=2,
                marker='o', markersize=5, label='Actual', zorder=6)

    # Vertical line separating evaluated vs forecast-only
    split_date = pd.Timestamp('2026-04-01')
    ax.axvline(split_date, color='#9E9E9E', linestyle=':', linewidth=1)

    # Shade forecast-only region
    ax.axvspan(split_date, FORECAST_MONTHS[-1] + pd.DateOffset(months=1),
               alpha=0.06, color='blue')

    # Baseline
    bl = baseline[baseline['force'] == force]['baseline'].values
    if len(bl):
        ax.axhline(bl[0], color='#9E9E9E', linestyle=':', linewidth=0.8)

    ax.set_title(force.replace(' Constabulary','').replace(' Police','').replace(' Service',''),
                 fontsize=8, fontweight='bold')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b%y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax.tick_params(axis='x', labelsize=6, rotation=30)
    ax.tick_params(axis='y', labelsize=6)

    # MAPE annotation
    mape_row = eval_df[eval_df['force'] == force]
    if not mape_row.empty:
        mape_val = mape_row['model_mape'].values[0]
        wins     = mape_row['model_wins'].values[0]
        color    = '#4CAF50' if wins else '#F44336'
        ax.text(0.98, 0.97, f'{mape_val:.1f}%', transform=ax.transAxes,
                ha='right', va='top', fontsize=7, color=color, fontweight='bold')

# Hide unused subplots
for j in range(i + 1, len(axes_flat)):
    axes_flat[j].set_visible(False)

handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, fontsize=9, bbox_to_anchor=(0.5, -0.01))
plt.tight_layout(rect=[0, 0.02, 1, 1])
plt.savefig('../outputs/all_forces_forecast_grid.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. H1 2026 forecast totals — budget allocation view

In [ ]:
# Aggregate full-year Jan–Dec forecast per force
full_year_forecast = (
    forecast_df.groupby('force')['forecast']
    .sum()
    .reset_index()
    .rename(columns={'forecast': 'forecast_total'})
)

# Actual Jan-Mar total per force
q1_actual = (
    forecast_df[forecast_df['actual'].notna()]
    .groupby('force')['actual']
    .sum()
    .reset_index()
    .rename(columns={'actual': 'q1_actual_total'})
)

# 2025 full-year actuals for YoY context
full_2025 = (
    force_monthly[force_monthly['month'].dt.year == 2025]
    .groupby('force')['count']
    .sum()
    .reset_index()
    .rename(columns={'count': 'full_2025_actual'})
)

budget_view = (
    full_year_forecast
    .merge(q1_actual, on='force', how='left')
    .merge(full_2025, on='force', how='left')
    .merge(eval_df[['force','model_mape','model_wins']], on='force', how='left')
)
budget_view['demand_share_pct'] = (
    budget_view['forecast_total'] / budget_view['forecast_total'].sum() * 100
).round(2)
budget_view['yoy_change_pct'] = (
    (budget_view['forecast_total'] - budget_view['full_2025_actual']) /
    budget_view['full_2025_actual'] * 100
).round(1)
budget_view = budget_view.sort_values('forecast_total', ascending=False).reset_index(drop=True)

print('=== Full-Year 2026 Forecast — Budget Allocation View ===')
display(
    budget_view[['force','forecast_total','q1_actual_total','full_2025_actual',
                 'demand_share_pct','yoy_change_pct','model_mape']]
    .round(1)
    .rename(columns={
        'force':'Force',
        'forecast_total':'2026 Forecast (Full Year)','q1_actual_total':'Q1 2026 Actual',
        'full_2025_actual':'2025 Actual (Full Year)','demand_share_pct':'Demand Share %',
        'yoy_change_pct':'YoY %','model_mape':'MAPE %'
    })
    .set_index('Force')
)

In [ ]:
# Demand share chart
top_n = budget_view.head(20)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle('Full-Year 2026 Demand Forecast — All Forces', fontsize=13, fontweight='bold')

axes[0].barh(top_n['force'].str.replace(' Constabulary','').str.replace(' Police','').str.replace(' Service',''),
             top_n['forecast_total'], color='#2196F3', edgecolor='white', linewidth=0.5)
axes[0].set_xlabel('Total forecast incidents (2026)')
axes[0].set_title('Top 20 Forces by Forecast Demand', fontsize=10)
axes[0].tick_params(axis='y', labelsize=8)
axes[0].invert_yaxis()

# YoY change
yoy = budget_view.dropna(subset=['yoy_change_pct']).sort_values('yoy_change_pct')
yoy_colors = ['#4CAF50' if v <= 0 else '#F44336' for v in yoy['yoy_change_pct']]
axes[1].barh(
    yoy['force'].str.replace(' Constabulary','').str.replace(' Police','').str.replace(' Service',''),
    yoy['yoy_change_pct'], color=yoy_colors, edgecolor='white', linewidth=0.5
)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('YoY change % vs 2025')
axes[1].set_title('Forecast YoY Change vs 2025\n(green = lower demand, red = higher)', fontsize=10)
axes[1].tick_params(axis='y', labelsize=7)

plt.tight_layout()
plt.savefig('../outputs/all_forces_2026_budget_view.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save outputs

In [ ]:
os.makedirs('../outputs', exist_ok=True)
forecast_df.to_parquet('../outputs/timesfm_all_forces_2026_forecasts.parquet', index=False)
forecast_df[['force','month','forecast']].assign(
    month=lambda d: d['month'].dt.strftime('%Y-%m'),
    forecast=lambda d: d['forecast'].clip(lower=0).round().astype(int)
).to_csv('../outputs/timesfm_all_forces_2026_forecasts.csv', index=False)
eval_df.to_csv('../outputs/timesfm_all_forces_q1_eval.csv', index=False)
budget_view.to_csv('../outputs/timesfm_all_forces_budget_view.csv', index=False)
print('Saved:')
print('  timesfm_all_forces_2026_forecasts.parquet — full Jan–Dec 2026 forecasts per force')
print('  timesfm_all_forces_2026_forecasts.csv     — same, CSV format')
print('  timesfm_all_forces_q1_eval.csv            — Jan–Mar evaluation metrics')
print('  timesfm_all_forces_budget_view.csv        — budget allocation summary table (full year)')